# MIMIC Ventilator Weaning Prediction with Trajectory Features
Compare baseline, summary statistics, trajectory, and combined features across models.

## Setup

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../../..'))
from notebook_utils import biomarker_summary_stats
from traj_features.backends.bootstrap import BootstrapTrajPS, BootstrapConfig
from traj_features.backends.bayes.classify import pos_flags_from_traj

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports successful')

WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


✓ Imports successful


## Load Data

In [2]:
pred_with_probs_path = '../../../results/mimic/ventilator/ventilator_prediction_dataset_with_probs.csv'
outcome_path = '../../../results/mimic/ventilator/ventilator_outcomes.csv'
ts_path = '../../../results/mimic/ventilator/pf_ratio_timeseries.csv'

df = pd.read_csv(pred_with_probs_path)
outcomes = pd.read_csv(outcome_path)
pf_ts = pd.read_csv(ts_path)

print(f'✓ Loaded prediction dataset with probs: {len(df):,} rows')
print(f'✓ Outcomes: {len(outcomes):,} rows')
print(f'✓ P/F ratio TS: {len(pf_ts):,} rows')

df = df.merge(outcomes[['hadm_id', 'time_day', 'target_weaning_success']], on=['hadm_id', 'time_day'], how='left')
df = df[df['target_weaning_success'].notna()].copy()
df['target_weaning_success'] = df['target_weaning_success'].astype(int)

if 'prob_improving' not in df.columns and {'prob_gradual_improvement', 'prob_rapid_improvement'}.issubset(df.columns):
    df['prob_improving'] = df['prob_gradual_improvement'].fillna(0) + df['prob_rapid_improvement'].fillna(0)

traj_cols = [c for c in df.columns if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]

print(f'Final dataset rows: {len(df):,}')
print(f'Outcome rate: {df["target_weaning_success"].mean():.1%}')
print(f'Trajectory columns: {len(traj_cols)}')

✓ Loaded prediction dataset with probs: 203,747 rows
✓ Outcomes: 24,341 rows
✓ P/F ratio TS: 58,255 rows
Final dataset rows: 24,339
Outcome rate: 6.3%
Trajectory columns: 4


In [3]:
# Bootstrap trajectory probabilities (cached)
bootstrap_path = '../../../results/mimic/ventilator/pf_trajectory_probs_bootstrap.csv'
label_map = {'nonprogression': 'prolonged_nonprogression', 'linear': 'linear_decline', 'nonlinear': 'nonlinear'}

if os.path.exists(bootstrap_path):
    pf_boot = pd.read_csv(bootstrap_path)
else:
    traj_input = pf_ts[['hadm_id', 'time_days', 'time_day', 'pf_ratio']].copy()
    traj_input = traj_input.rename(columns={'hadm_id': 'patientid', 'pf_ratio': 'lab_value'})
    traj_input = traj_input.dropna(subset=['lab_value']).sort_values(by=['patientid', 'time_days'])
    model = BootstrapTrajPS(BootstrapConfig(
        window_years=3.0,
        n_bootstrap=200,
        smoothing=None,
        min_points_per_window=4,
        grid_freq=2,
        flat_thr=10.0,
        decline_thr=30.0,
        nonlinear_gap=20.0,
        pids='patientid',
        values='lab_value',
        time_col='time_days',
        windowing_col='time_day',
        n_jobs=-1,
        traj_types=('prolonged_nonprogression', 'linear_decline', 'nonlinear'),
        class_func=pos_flags_from_traj,
        label_map=label_map,
        progressbar=False
    ))
    pf_boot = model.embed(traj_input)
    pf_boot = pf_boot.rename(columns={'patientid': 'hadm_id'})
    pf_boot.to_csv(bootstrap_path, index=False)

if 'patientid' in pf_boot.columns:
    pf_boot = pf_boot.rename(columns={'patientid': 'hadm_id'})

pf_boot = pf_boot.rename(columns={
    'trajtype_prolonged_nonprogression_prob': 'prob_stable_boot',
    'trajtype_linear_decline_prob': 'prob_gradual_improvement_boot',
    'trajtype_nonlinear_prob': 'prob_rapid_improvement_boot',
})

df = df.merge(
    pf_boot[['hadm_id', 'time_day', 'prob_stable_boot', 'prob_gradual_improvement_boot', 'prob_rapid_improvement_boot']],
    on=['hadm_id', 'time_day'],
    how='left'
)

if {'prob_gradual_improvement_boot', 'prob_rapid_improvement_boot'}.issubset(df.columns):
    df['prob_improving_boot'] = df['prob_gradual_improvement_boot'].fillna(0) + df['prob_rapid_improvement_boot'].fillna(0)

print('✓ Added bootstrap trajectory probabilities')

WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.configdefaults): g++ n

✓ Added bootstrap trajectory probabilities


## Feature Sets

In [4]:
pf_summary = biomarker_summary_stats(pf_ts, value_col='pf_ratio', lookback_days=3)
df = df.merge(pf_summary, on=['hadm_id', 'time_day'], how='left')

exclude_cols = {'hadm_id', 'time_day', 'charttime', 'admittime', 'dischtime', 'target_weaning_success', 'subject_id',
 'los_days', 'hospital_expire_flag', 'in_hospital_mortality', 'icu_expire_flag', 'icu_mortality'}

numeric_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
traj_cols = [c for c in numeric_cols if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]
traj_cols_boot = [c for c in traj_cols if c.endswith('_boot')]
traj_cols_bayes = [c for c in traj_cols if not c.endswith('_boot')]
summary_cols = [c for c in numeric_cols if c.endswith('_3d') or '_trend_' in c or '_change_' in c]
base_cols = [c for c in numeric_cols if c not in traj_cols and c not in summary_cols]

static_name_tokens = ['age', 'gender', 'sex', 'baseline', 'admit', 'admission', 'ethnicity', 'race', 'height', 'weight', 'bmi']
static_cols = [c for c in base_cols if any(tok in c.lower() for tok in static_name_tokens)]
dynamic_cols = [c for c in base_cols if c not in static_cols]
static_dynamic_cols = static_cols + dynamic_cols

feature_sets = {
    'Trajectory Only': traj_cols_bayes,
    'Summary Stats Only': summary_cols,
    'Trajectory + Summary Stats': traj_cols_bayes + summary_cols,
    'Static Only': static_cols,
    'Trajectory + Static': traj_cols_bayes + static_cols,
    'Summary Stats + Static': summary_cols + static_cols,
    'Trajectory + Summary Stats + Static': traj_cols_bayes + summary_cols + static_cols,
    'Trajectory Only (Bootstrap)': traj_cols_boot,
    'Trajectory + Summary Stats (Bootstrap)': traj_cols_boot + summary_cols,
    'Trajectory + Static (Bootstrap)': traj_cols_boot + static_cols,
    'Trajectory + Summary Stats + Static (Bootstrap)': traj_cols_boot + summary_cols + static_cols,
}

# Filter feature sets to remove unavailable columns and add dynamic combinations
for set_name, cols in list(feature_sets.items()):
    available = [c for c in cols if c in df.columns]
    removed = [c for c in cols if c not in df.columns]
    if removed:
        print(f"Feature set '{set_name}': removed {len(removed)} missing columns: {removed}")
    feature_sets[set_name] = available

if len(dynamic_cols) > 0:
    feature_sets['Static + Dynamic'] = [c for c in static_dynamic_cols if c in df.columns]
    feature_sets['Static + Dynamic + Trajectory'] = [c for c in static_dynamic_cols + traj_cols_bayes if c in df.columns]
    feature_sets['Static + Dynamic + Trajectory (Bootstrap)'] = [c for c in static_dynamic_cols + traj_cols_boot if c in df.columns]
    feature_sets['Static + Dynamic + Summary'] = [c for c in static_dynamic_cols + summary_cols if c in df.columns]
    feature_sets['Static + Dynamic + Summary + Trajectory'] = [c for c in static_dynamic_cols + summary_cols + traj_cols_bayes if c in df.columns]
    feature_sets['Static + Dynamic + Summary + Trajectory (Bootstrap)'] = [c for c in static_dynamic_cols + summary_cols + traj_cols_boot if c in df.columns]

for name, cols in feature_sets.items():
    print(f'{name}: {len(cols)} features')

 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number 

## Model Comparison

In [5]:
df = df.drop_duplicates(subset=['hadm_id', 'time_day'])

In [6]:
df.shape

(20813, 109)

In [ ]:
models_to_evaluate = {
    'LogReg': lambda: LogisticRegression(max_iter=300, n_jobs=-1),
    'RF': lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=920),
    'HGB': lambda: HistGradientBoostingClassifier(random_state=920),
    'XGB': lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, eval_metric='logloss', random_state=920),
}

n_repeats = 10
n_splits = 5
results = {model_name: {} for model_name in models_to_evaluate.keys()}
y = df['target_weaning_success']
groups = df['hadm_id']

for model_name, model_fn in models_to_evaluate.items():
    dataset_model = df.copy()
    if len(traj_cols) > 0:
        dataset_model[traj_cols] = dataset_model.groupby('hadm_id')[traj_cols].ffill(limit=2)
    if model_name not in ['XGB', 'HGB']:
        if len(traj_cols) > 0:
            dataset_model[traj_cols] = dataset_model[traj_cols].fillna(0)
        if len(summary_cols) > 0:
            dataset_model[summary_cols] = dataset_model[summary_cols].fillna(0)

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            continue
        fold_metrics = {'roc_auc': [], 'avg_precision': [], 'y_true': [], 'y_pred': []}
        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920 + repeat).permutation(len(dataset_model))
            dataset_repeat = dataset_model.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_splits)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                if model_name not in ['XGB', 'HGB']:
                    imputer = SimpleImputer(strategy='median')
                    X_train_imputed = imputer.fit_transform(X_train)
                    X_test_imputed = imputer.transform(X_test)
                else:
                    X_train_imputed = X_train
                    X_test_imputed = X_test

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                model = model_fn()
                model.fit(X_train_scaled, y_train)

                if hasattr(model, 'predict_proba'):
                    probs = model.predict_proba(X_test_scaled)[:, 1]
                else:
                    probs = model.decision_function(X_test_scaled)

                fold_metrics['roc_auc'].append(roc_auc_score(y_test, probs))
                fold_metrics['avg_precision'].append(average_precision_score(y_test, probs))
                fold_metrics['y_true'].extend(y_test.tolist())
                fold_metrics['y_pred'].extend(probs.tolist())

        results[model_name][feature_set_name] = fold_metrics
        print(f"{model_name} | {feature_set_name}: AUROC {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}, AUPRC {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}")

summary_rows = []
for model_name, model_results in results.items():
    for feature_set_name, metrics in model_results.items():
        summary_rows.append({
            'model': model_name,
            'feature_set': feature_set_name,
            'auroc_mean': np.mean(metrics['roc_auc']),
            'auroc_std': np.std(metrics['roc_auc']),
            'auprc_mean': np.mean(metrics['avg_precision']),
            'auprc_std': np.std(metrics['avg_precision'])
        })

results_df = pd.DataFrame(summary_rows)
results_df

LogReg | Trajectory Only: AUROC 0.640 ± 0.009, AUPRC 0.104 ± 0.003
LogReg | Summary Stats Only: AUROC 0.741 ± 0.016, AUPRC 0.227 ± 0.011
LogReg | Trajectory + Summary Stats: AUROC 0.751 ± 0.011, AUPRC 0.234 ± 0.006
LogReg | Static Only: AUROC 0.626 ± 0.011, AUPRC 0.119 ± 0.008
LogReg | Trajectory + Static: AUROC 0.699 ± 0.009, AUPRC 0.154 ± 0.007
LogReg | Summary Stats + Static: AUROC 0.746 ± 0.016, AUPRC 0.230 ± 0.009
LogReg | Trajectory + Summary Stats + Static: AUROC 0.755 ± 0.013, AUPRC 0.238 ± 0.009
LogReg | Trajectory Only (Bootstrap): AUROC 0.646 ± 0.008, AUPRC 0.103 ± 0.003
LogReg | Trajectory + Summary Stats (Bootstrap): AUROC 0.751 ± 0.011, AUPRC 0.233 ± 0.009
LogReg | Trajectory + Static (Bootstrap): AUROC 0.706 ± 0.008, AUPRC 0.156 ± 0.007
LogReg | Trajectory + Summary Stats + Static (Bootstrap): AUROC 0.757 ± 0.012, AUPRC 0.236 ± 0.008
LogReg | Static + Dynamic: AUROC 0.707 ± 0.013, AUPRC 0.179 ± 0.013
LogReg | Trajectory + Static + Dynamic: AUROC 0.735 ± 0.010, AUPRC 0.20

## Compare Model Performance

In [ ]:
print("=" * 100)
print("MODEL COMPARISON - All Feature Sets")
print("=" * 100)

summary_df = results_df.copy()
summary_df['ROC-AUC'] = summary_df.apply(lambda r: f"{r.auroc_mean:.3f} ± {r.auroc_std:.3f}", axis=1)
summary_df['AUPR'] = summary_df.apply(lambda r: f"{r.auprc_mean:.3f} ± {r.auprc_std:.3f}", axis=1)

for model_name in summary_df['model'].unique():
    print(f"{model_name}:")
    model_df = summary_df[summary_df['model'] == model_name]
    print(model_df[['feature_set', 'ROC-AUC', 'AUPR']].to_string(index=False))
    print("\n")

print("\n" + "=" * 100)
print("TOP 10 CONFIGURATIONS BY AUROC")
print("=" * 100)
top10 = summary_df.sort_values('auroc_mean', ascending=False).head(10)
print(top10[['model', 'feature_set', 'ROC-AUC', 'AUPR']].to_string(index=False))

print("\n" + "=" * 100)
print("BEST MODEL PER FEATURE SET")
print("=" * 100)
for feature_set_name in summary_df['feature_set'].unique():
    best = summary_df[summary_df['feature_set'] == feature_set_name].sort_values('auroc_mean', ascending=False).iloc[0]
    print(f"  {feature_set_name:45s}: {best['model']:10s} (AUROC={best['auroc_mean']:.3f}, AUPR={best['auprc_mean']:.3f})")

## Visualize Results

In [ ]:
# ROC and PR curves for each model (key feature sets)
for model_name in results.keys():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f'{model_name} - Key Feature Sets Comparison', fontsize=14, fontweight='bold')

    key_sets = [
        'Trajectory Only',
        'Summary Stats Only',
        'Trajectory + Summary Stats',
        'Static + Dynamic',
        'Trajectory + Summary Stats + Static + Dynamic'
    ]

    colors = plt.cm.tab10(np.linspace(0, 1, len(key_sets)))

    for idx, name in enumerate(key_sets):
        if name not in results[model_name]:
            continue
        metrics = results[model_name][name]
        fpr, tpr, _ = roc_curve(metrics['y_true'], metrics['y_pred'])
        auc = np.mean(metrics['roc_auc'])
        ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=colors[idx], linewidth=2)

        precision, recall, _ = precision_recall_curve(metrics['y_true'], metrics['y_pred'])
        ap = np.mean(metrics['avg_precision'])
        ax2.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=colors[idx], linewidth=2)

    ax1.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.legend(loc='lower right', fontsize=9)
    ax1.grid(True, alpha=0.3)

    baseline = y.mean()
    ax2.axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.3f})', linewidth=1)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Model Comparison Summary (boxplots)
for model_name in results.keys():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'{model_name} - Performance Across All Feature Sets', fontsize=14, fontweight='bold')

    feature_set_names = list(results[model_name].keys())
    roc_data = [results[model_name][name]['roc_auc'] for name in feature_set_names]
    ap_data = [results[model_name][name]['avg_precision'] for name in feature_set_names]

    bp1 = ax1.boxplot(roc_data, labels=feature_set_names, patch_artist=True)
    ax1.set_ylabel('ROC-AUC')
    ax1.set_title('ROC-AUC Across All Feature Sets')
    ax1.set_xticklabels(feature_set_names, rotation=45, ha='right')
    ax1.grid(alpha=0.3, axis='y')

    bp2 = ax2.boxplot(ap_data, labels=feature_set_names, patch_artist=True)
    ax2.set_ylabel('Average Precision')
    ax2.set_title('AUPR Across All Feature Sets')
    ax2.set_xticklabels(feature_set_names, rotation=45, ha='right')
    ax2.grid(alpha=0.3, axis='y')

    for bp in [bp1, bp2]:
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')

    plt.tight_layout()
    plt.show()

# Summary bar plots (AUROC/AUPRC means)
summary_plot_df = results_df.copy()
plt.figure(figsize=(12, 5))
sns.catplot(data=summary_plot_df, x='feature_set', y='auroc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC Ventilator: AUROC by Feature Set and Model')
plt.ylabel('AUROC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

sns.catplot(data=summary_plot_df, x='feature_set', y='auprc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC Ventilator: AUPRC by Feature Set and Model')
plt.ylabel('AUPRC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
summary = results_df.pivot_table(index='feature_set', columns='model', values='auroc_mean')
summary

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='feature_set', y='auroc_mean', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUROC by Feature Set and Model')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='feature_set', y='auprc_mean', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUPRC by Feature Set and Model')
plt.tight_layout()
plt.show()